# 062 — Model Comparison (beta-NLL vs. gaussian-NLL, unet_v2 vs. unet)

Qualitative counterpart of `032_evaluation_v2.ipynb`'s metrics tables: for one real test image, compares hidden-detail signals side by side.

- **Part A**: for each of the four NLL architectures, `gaussian_nll` (`021_training_nll.ipynb`) vs. `beta_nll` (`022_training_v2.ipynb`) — raw delta and learned z-score, via `scripts.visualization_nll.plot_signal_comparison`.
- **Part B**: `unet` vs. `unet_v2` — raw delta and structural delta (`scripts.delta_analysis.analyze_delta`), the same comparison tool reused for the two deterministic architectures.

Mirrors `060_model_comparison_nll.ipynb`. Missing checkpoints are skipped with a message rather than failing the notebook.

In [ ]:
import sys
from pathlib import Path

project_root = Path().absolute()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from PIL import Image

from scripts.config import settings
from scripts.dataset import pad_to_multiple
from scripts.delta_analysis import analyze_delta
from scripts.trainer import load_model
from scripts.trainer_nll import load_model_nll
from scripts.visualization_nll import plot_signal_comparison

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs available: {gpus}")

## 1. Select image

Falls back to a synthetic `(rgb, real_ir)` pair (same layout as `050_delta_analysis.ipynb`/`060_model_comparison_nll.ipynb`) if `data/test/` has no real pairs, so the notebook stays runnable end to end without test data.

In [ ]:
IMAGE_STEM = None  # e.g. "green" — pick a specific image from data/test/rgb/
# None = first available

TEST_RGB_DIR = project_root / "data" / "test" / "rgb"
TEST_IR_DIR = project_root / "data" / "test" / "ir"

rgb_paths = sorted(TEST_RGB_DIR.glob("*.jpg")) if TEST_RGB_DIR.exists() else []
test_pairs = [
    (p, TEST_IR_DIR / p.name) for p in rgb_paths if (TEST_IR_DIR / p.name).exists()
]
print(f"Available pairs: {[p.stem for p, _ in test_pairs]}")


def make_synthetic_pair(size: int = 256, seed: int = 0):
    """Build a synthetic (rgb, real_ir) pair for a dry run."""
    rng = np.random.default_rng(seed)
    x, y = np.meshgrid(np.linspace(0, 6 * np.pi, size), np.linspace(0, 6 * np.pi, size))
    texture = 0.5 + 0.25 * np.sin(x) * np.cos(y)
    real = np.clip(texture + rng.normal(0, 0.02, size=(size, size)), 0.0, 1.0).astype(
        np.float32
    )
    rgb = np.stack([real] * 3, axis=-1)
    return rgb, real


def select_pair(pairs, stem):
    """Return the (rgb, ir) path pair matching stem, or the first pair if None."""
    if stem is None:
        return pairs[0]
    matches = [pair for pair in pairs if pair[0].stem == stem]
    if not matches:
        available = [p.stem for p, _ in pairs]
        raise ValueError(f"IMAGE_STEM='{stem}' not found. Available: {available}")
    return matches[0]


USE_SYNTHETIC_DATA = not test_pairs
print(f"USE_SYNTHETIC_DATA = {USE_SYNTHETIC_DATA}")

if USE_SYNTHETIC_DATA:
    rgb, ir_real = make_synthetic_pair()
    pair_label = "synthetic"
else:
    rgb_path, ir_path = select_pair(test_pairs, IMAGE_STEM)
    rgb = np.array(Image.open(rgb_path).convert("RGB")).astype(np.float32) / 255.0
    ir_real = np.array(Image.open(ir_path).convert("L")).astype(np.float32) / 255.0
    pair_label = rgb_path.stem

print(f"Pair: {pair_label} | shape: {ir_real.shape}")

## 2. Part A — beta-NLL vs. gaussian-NLL, per architecture

For each architecture with both checkpoints available: raw delta `|real_IR - mu|` and learned z-score `(real_IR - mu) / sigma`, side by side for the two loss variants.

In [ ]:
ARCHS_NLL = ["unet_nll", "resunet_nll", "attention_unet_nll", "efficientnet_unet_nll"]
BETA_MODEL_DIR = settings.MODELS_DIR / "nll_beta"
BETA = 0.5
Z_VMAX = 4.0


def _predict_mu_sigma(model, rgb, ir_real):
    padded, _ = pad_to_multiple(tf.constant(rgb), multiple=settings.PATCH_MULTIPLE)
    pred_full = model.predict(padded[tf.newaxis, ...], verbose=0)[0]
    h, w = ir_real.shape
    mu = pred_full[:h, :w, 0]
    log_var = pred_full[:h, :w, 1]
    sigma = np.exp(0.5 * log_var)
    return mu, sigma


for arch in ARCHS_NLL:
    try:
        model_g = load_model_nll(
            arch, model_dir=settings.MODELS_DIR, loss_name="gaussian_nll"
        )
    except (FileNotFoundError, OSError) as exc:
        print(f"Skipping '{arch}' (gaussian_nll): {exc}")
        continue
    try:
        model_b = load_model_nll(
            arch, model_dir=BETA_MODEL_DIR, loss_name="beta_nll", beta=BETA
        )
    except (FileNotFoundError, OSError) as exc:
        print(f"Skipping '{arch}' (beta_nll): {exc}")
        continue

    mu_g, sigma_g = _predict_mu_sigma(model_g, rgb, ir_real)
    mu_b, sigma_b = _predict_mu_sigma(model_b, rgb, ir_real)

    raw_delta = {
        f"{arch} (gaussian)": np.abs(ir_real - mu_g),
        f"{arch} (beta)": np.abs(ir_real - mu_b),
    }
    zscore = {
        f"{arch} (gaussian)": (ir_real - mu_g) / (sigma_g + 1e-8),
        f"{arch} (beta)": (ir_real - mu_b) / (sigma_b + 1e-8),
    }

    fig = plot_signal_comparison(
        ir_real,
        raw_delta,
        title=f"Raw delta — {arch} — {pair_label}",
        vrange=(0.0, 1.0),
    )
    plt.show()

    fig = plot_signal_comparison(
        ir_real,
        zscore,
        title=f"Learned z-score — {arch} — {pair_label}",
        vrange=(-Z_VMAX, Z_VMAX),
    )
    plt.show()

## 3. Part B — unet vs. unet_v2

Raw delta and structural delta (`scripts.delta_analysis.analyze_delta`) for the two deterministic architectures — both purely a function of the predicted IR, no `sigma`/z-score involved (deterministic models have no uncertainty head).

In [ ]:
raw_delta_det: dict[str, np.ndarray] = {}
structural_delta_det: dict[str, np.ndarray] = {}

for arch in ["unet", "unet_v2"]:
    try:
        model = load_model(arch, model_dir=settings.MODELS_DIR)
    except (FileNotFoundError, OSError) as exc:
        print(f"Skipping '{arch}': {exc}")
        continue

    padded, _ = pad_to_multiple(tf.constant(rgb), multiple=settings.PATCH_MULTIPLE)
    pred_full = model.predict(padded[tf.newaxis, ...], verbose=0)[0]
    h, w = ir_real.shape
    ir_pred = pred_full[:h, :w, 0]

    result = analyze_delta(ir_real, ir_pred, window_size=11, zone_size=32)
    raw_delta_det[arch] = np.abs(ir_real - ir_pred)
    structural_delta_det[arch] = result.structural_delta

    print(f"{arch}: done")

if raw_delta_det:
    fig = plot_signal_comparison(
        ir_real,
        raw_delta_det,
        title=f"Raw delta — unet vs. unet_v2 — {pair_label}",
        vrange=(0.0, 1.0),
    )
    plt.show()

if structural_delta_det:
    fig = plot_signal_comparison(
        ir_real,
        structural_delta_det,
        title=f"Structural delta — unet vs. unet_v2 — {pair_label}",
        vrange=(0.0, 1.0),
    )
    plt.show()